# Xarray-spatial
### User Guide: Emerging Hot Spot Analysis
-----

Emerging hot spot analysis answers the question: **how are spatial clusters changing over time?**

A single snapshot can tell you *where* hot spots are right now, but it can't tell you whether they're growing, shrinking, just appeared, or have been there all along. By analysing a stack of time steps together, `emerging_hotspots` classifies every pixel into one of **17 trend categories** that describe its trajectory.

The method combines two well-known statistical techniques:

1. **Getis-Ord Gi\*** (per time step) — identifies statistically significant spatial clusters of high or low values.
2. **Mann-Kendall trend test** (across time) — detects whether a pixel's cluster intensity is trending up, down, or staying flat.

This is useful for satellite imagery captured every few days, temperature grids over seasons, crime density maps over months, or any raster that evolves over time.

-----

[Setup](#Setup): Imports and helper functions

[Building a Synthetic Dataset](#Building-a-Synthetic-Dataset): Create a 3D raster with known hot and cold spot patterns

[Running the Analysis](#Running-the-Analysis): Call `emerging_hotspots` and explore the output

[Understanding the Categories](#Understanding-the-Categories): What each of the 17 trend codes means

[Visualising the Results](#Visualising-the-Results): Map the trend categories with colour

[Inspecting Individual Pixels](#Inspecting-Individual-Pixels): Drill into the time series at specific locations

[Boundary Modes](#Boundary-Modes): How edge handling affects results

[Working with Dask](#Working-with-Dask): Scaling to large datasets

-----------

## Setup

We'll use **matplotlib** for all visualizations (raster rendering, time-series line plots, and the category legend) and **xrspatial** for the analysis itself.

In [ ]:
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm

from xrspatial.convolution import circle_kernel, calc_cellsize
from xrspatial.emerging_hotspots import emerging_hotspots

## Building a Synthetic Dataset

To see every category in action we'll build a **20-step, 100×100** raster with several planted signals on top of random background noise. Each signal is designed to trigger a specific trend category.

| Region | Signal | Expected category |
|--------|--------|-------------------|
| Centre-left | Strong positive values at *every* step | **Persistent Hot Spot** (4) |
| Centre-right | Strong negative values at every step | **Persistent Cold Spot** (-4) |
| Top-left | Positive signal only in the *last* step | **New Hot Spot** (1) |
| Top-right | Positive signal growing over time (ramp) | **Intensifying Hot Spot** (3) |
| Bottom-left | Positive signal fading over time | **Diminishing Hot Spot** (5) |
| Bottom-right | Positive at every step except the last | **Historical Hot Spot** (8) |

In [ ]:
n_times, ny, nx = 20, 100, 100
rng = np.random.default_rng(42)

# Background noise
data = rng.standard_normal((n_times, ny, nx)).astype(np.float32)

# --- Persistent hot spot (centre-left) ---
data[:, 40:50, 20:30] += 30.0

# --- Persistent cold spot (centre-right) ---
data[:, 40:50, 70:80] -= 30.0

# --- New hot spot (top-left): signal only at the final step ---
data[-1, 10:18, 10:18] += 40.0

# --- Intensifying hot spot (top-right): ramp from weak to strong ---
for t in range(n_times):
    strength = 5.0 + 30.0 * (t / (n_times - 1))  # 5 -> 35
    data[t, 10:18, 75:83] += strength

# --- Diminishing hot spot (bottom-left): ramp from strong to weak ---
for t in range(n_times):
    strength = 35.0 - 30.0 * (t / (n_times - 1))  # 35 -> 5
    data[t, 80:88, 10:18] += strength

# --- Historical hot spot (bottom-right): strong everywhere except last ---
data[:-1, 80:88, 75:83] += 30.0

raster = xr.DataArray(data, dims=['time', 'y', 'x'])
print(raster)

Let's peek at the first and last time steps side by side to see the planted patterns. Bright pixels are high values, dark pixels are low values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vmin, vmax = np.nanpercentile(data, [2, 98])
for ax, t, label in zip(axes, [0, -1], ['First time step (t=0)', 'Last time step (t=19)']):
    im = ax.imshow(data[t], cmap='RdBu_r', vmin=vmin, vmax=vmax, origin='upper')
    ax.set_title(label, fontsize=13)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.colorbar(im, ax=axes, label='Value', shrink=0.8)
fig.suptitle('Synthetic raster — first vs. last time step', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## Running the Analysis

`emerging_hotspots` needs two things:

1. A **3-D DataArray** with dimensions `(time, y, x)`.
2. A **2-D kernel** defining the spatial neighbourhood (just like the regular `hotspots` function).

We'll use a simple 5×5 ones kernel here. In practice you'd choose a kernel size that matches the spatial scale of the clusters you're looking for — `circle_kernel` and `annulus_kernel` from `xrspatial.convolution` are convenient for this.

In [ ]:
kernel = np.ones((5, 5), dtype=np.float32)

result = emerging_hotspots(raster, kernel)
result

The result is an `xarray.Dataset` with five variables:

| Variable | Shape | Description |
|----------|-------|-------------|
| `category` | `(y, x)` | The trend classification code (−8 to 8) |
| `gi_zscore` | `(time, y, x)` | Gi\* z-score at each time step |
| `gi_bin` | `(time, y, x)` | Confidence bin (±90, ±95, ±99, or 0) |
| `trend_zscore` | `(y, x)` | Mann-Kendall Z statistic |
| `trend_pvalue` | `(y, x)` | Mann-Kendall two-sided p-value |

## Understanding the Categories

Each pixel is assigned one of **17 categories** (8 hot, 8 cold, plus "no pattern"). The hot-spot categories are:

| Code | Name | What it means |
|:----:|------|---------------|
| **1** | **New** | A hot spot *only* at the final time step — it just appeared. |
| **2** | **Consecutive** | A hot spot for the last N consecutive steps (N ≥ 2), but never before that. |
| **3** | **Intensifying** | A hot spot for ≥ 90% of steps including the last, and the Mann-Kendall test shows a **significant upward trend**. Getting hotter. |
| **4** | **Persistent** | A hot spot for ≥ 90% of steps including the last, with **no significant trend**. Stable. |
| **5** | **Diminishing** | A hot spot for ≥ 90% of steps including the last, but with a **significant downward trend**. Cooling off. |
| **6** | **Sporadic** | A hot spot at the final step, but in fewer than 90% of steps, and never a cold spot. On-and-off. |
| **7** | **Oscillating** | A hot spot at the final step, but was a cold spot at least once. Flipping between extremes. |
| **8** | **Historical** | A hot spot for ≥ 90% of steps, but **not** at the final step. It used to be hot but isn't anymore. |

Codes **−1 through −8** are the exact mirror for **cold spots** (swap "high" and "low"). Code **0** means no significant pattern was detected.

## Visualising the Results

Let's build a colour map that gives each category a distinct, intuitive colour. Hot-spot categories use warm tones (reds/oranges), cold-spot categories use cool tones (blues), and "no pattern" is light grey.

In [ ]:
# Category metadata: (code, label, colour)
CATEGORY_INFO = [
    (-8, 'Historical Cold',   '#08306b'),
    (-7, 'Oscillating Cold',  '#08519c'),
    (-6, 'Sporadic Cold',     '#2171b5'),
    (-5, 'Diminishing Cold',  '#4292c6'),
    (-4, 'Persistent Cold',   '#6baed6'),
    (-3, 'Intensifying Cold', '#9ecae1'),
    (-2, 'Consecutive Cold',  '#c6dbef'),
    (-1, 'New Cold',          '#deebf7'),
    ( 0, 'No Pattern',        '#d9d9d9'),
    ( 1, 'New Hot',           '#fee0d2'),
    ( 2, 'Consecutive Hot',   '#fcbba1'),
    ( 3, 'Intensifying Hot',  '#fc9272'),
    ( 4, 'Persistent Hot',    '#fb6a4a'),
    ( 5, 'Diminishing Hot',   '#ef3b2c'),
    ( 6, 'Sporadic Hot',      '#cb181d'),
    ( 7, 'Oscillating Hot',   '#a50f15'),
    ( 8, 'Historical Hot',    '#67000d'),
]

codes  = [c for c, _, _ in CATEGORY_INFO]
labels = [l for _, l, _ in CATEGORY_INFO]
colors = [c for _, _, c in CATEGORY_INFO]

# Build a discrete colormap
cmap = ListedColormap(colors)
boundaries = [c - 0.5 for c in codes] + [codes[-1] + 0.5]
norm = BoundaryNorm(boundaries, cmap.N)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

cat = result['category'].values
im = ax.imshow(cat, cmap=cmap, norm=norm, origin='upper', interpolation='nearest')
ax.set_title('Emerging Hot Spot Categories', fontsize=15)
ax.set_xlabel('x')
ax.set_ylabel('y')

# Build a legend with only the categories that actually appear
unique_cats = sorted(np.unique(cat[~np.isnan(cat.astype(float))]).astype(int))
legend_patches = []
for code, label, color in CATEGORY_INFO:
    if code in unique_cats:
        legend_patches.append(mpatches.Patch(facecolor=color, edgecolor='gray',
                                             label=f'{code:+d}  {label}'))

ax.legend(handles=legend_patches, loc='center left', bbox_to_anchor=(1.02, 0.5),
          fontsize=10, title='Category', title_fontsize=11,
          frameon=True, fancybox=True)

plt.tight_layout()
plt.show()

You should see the planted signals show up clearly:

- **Centre-left** block in warm red → Persistent Hot (4)
- **Centre-right** block in blue → Persistent Cold (−4)
- **Top-left** block in pale pink → New Hot (1)
- **Top-right** block in salmon → Intensifying Hot (3)
- **Bottom-left** block in deeper red → Diminishing Hot (5)
- **Bottom-right** block in dark maroon → Historical Hot (8)
- Everything else in grey → No Pattern (0)

### Mann-Kendall Trend Map

The `trend_zscore` variable shows *how strongly* each pixel is trending, independent of the discrete categories. Positive values mean increasing intensity over time, negative means decreasing. Let's map it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trend Z-score
tz = result['trend_zscore'].values
vlim = np.nanpercentile(np.abs(tz), 99)
im0 = axes[0].imshow(tz, cmap='RdBu_r', vmin=-vlim, vmax=vlim, origin='upper')
axes[0].set_title('Mann-Kendall Trend Z-score', fontsize=13)
fig.colorbar(im0, ax=axes[0], shrink=0.8)

# Trend p-value (log scale for visibility)
tp = result['trend_pvalue'].values
tp_log = -np.log10(np.clip(tp, 1e-20, 1.0))  # higher = more significant
im1 = axes[1].imshow(tp_log, cmap='magma', origin='upper')
axes[1].set_title('Trend Significance  (−log₁₀ p-value)', fontsize=13)
fig.colorbar(im1, ax=axes[1], shrink=0.8)

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()

The intensifying hot spot (top-right) has a large positive Z-score and very small p-value, confirming a strong upward trend. The diminishing hot spot (bottom-left) shows a strong negative Z-score — still a hot spot, but losing intensity.

## Inspecting Individual Pixels

The `gi_zscore` and `gi_bin` variables let you drill into the time series at any location. Let's compare three representative pixels.

In [ ]:
# Pick pixels from three planted regions
pixels = {
    'Persistent Hot (4)':    (45, 25),
    'Intensifying Hot (3)':  (14, 79),
    'Historical Hot (8)':    (84, 79),
}

fig, axes = plt.subplots(len(pixels), 1, figsize=(12, 3.2 * len(pixels)),
                         sharex=True)

time_steps = np.arange(n_times)

for ax, (label, (py, px)) in zip(axes, pixels.items()):
    zs = result['gi_zscore'].values[:, py, px]
    bins = result['gi_bin'].values[:, py, px]
    cat_code = int(result['category'].values[py, px])

    # Colour each bar by its confidence bin
    bar_colors = []
    for b in bins:
        if b >= 90:
            bar_colors.append('#fb6a4a')   # hot
        elif b <= -90:
            bar_colors.append('#6baed6')   # cold
        else:
            bar_colors.append('#d9d9d9')   # not significant

    ax.bar(time_steps, zs, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axhline(1.96, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(-1.96, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.set_ylabel('Gi* z-score')
    ax.set_title(f'{label}   (pixel y={py}, x={px},  category={cat_code:+d})',
                 fontsize=12)

axes[-1].set_xlabel('Time step')
axes[-1].set_xticks(time_steps)

# Shared legend
legend_items = [
    mpatches.Patch(facecolor='#fb6a4a', edgecolor='gray', label='Significant hot'),
    mpatches.Patch(facecolor='#6baed6', edgecolor='gray', label='Significant cold'),
    mpatches.Patch(facecolor='#d9d9d9', edgecolor='gray', label='Not significant'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Gi* z-score time series at selected pixels', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Reading these charts:

- **Persistent Hot** — every bar is red (significant hot spot) and roughly the same height. No trend, just steady.
- **Intensifying Hot** — bars start shorter and grow taller. The z-scores increase over time, which is what the Mann-Kendall test picks up.
- **Historical Hot** — red bars for most of the series, then drops to grey at the end. It *was* a hot spot, but not anymore.

### Gi\* Confidence Bins Over Time

We can also visualise the `gi_bin` variable as a heatmap across all pixels and time steps. This shows the *spatial pattern of significance* evolving through time.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

steps_to_show = [0, 6, 13, 19]
bin_cmap = ListedColormap(['#08519c', '#6baed6', '#c6dbef',
                           '#d9d9d9',
                           '#fcbba1', '#fb6a4a', '#cb181d'])
bin_bounds = [-99.5, -95.5, -90.5, -0.5, 0.5, 90.5, 95.5, 99.5]
bin_norm = BoundaryNorm(bin_bounds, bin_cmap.N)

for ax, t in zip(axes, steps_to_show):
    ax.imshow(result['gi_bin'].values[t], cmap=bin_cmap, norm=bin_norm,
              origin='upper', interpolation='nearest')
    ax.set_title(f't = {t}', fontsize=12)
    ax.set_xlabel('x')

axes[0].set_ylabel('y')

# Legend
bin_labels = ['-99%', '-95%', '-90%', 'n.s.', '+90%', '+95%', '+99%']
bin_legend = [mpatches.Patch(facecolor=c, edgecolor='gray', label=l)
              for c, l in zip(bin_cmap.colors, bin_labels)]
fig.legend(handles=bin_legend, loc='lower center', ncol=7, fontsize=9,
           bbox_to_anchor=(0.5, -0.06))

fig.suptitle('Gi* confidence bins at selected time steps', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Notice how the top-left block (New Hot Spot) only lights up at t=19, while the bottom-right block (Historical Hot Spot) is bright at t=0 through t=13 but fades by t=19.

## Boundary Modes

Like the regular `hotspots` function, `emerging_hotspots` supports four boundary modes that control how the kernel handles pixels near the edges of the raster:

| Mode | Behaviour |
|------|-----------|
| `'nan'` (default) | Neighbours outside the raster are treated as NaN. Edge pixels get NaN z-scores. |
| `'nearest'` | The nearest edge value is repeated outward. |
| `'reflect'` | Values are mirrored at the boundary. |
| `'wrap'` | Values wrap around (periodic / toroidal). |

The default `'nan'` mode is the most conservative — it avoids making assumptions about what's beyond the edge. The other modes are useful when you want every pixel to receive a valid z-score.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, mode in zip(axes, ['nan', 'nearest', 'reflect', 'wrap']):
    ds_mode = emerging_hotspots(raster, kernel, boundary=mode)
    ax.imshow(ds_mode['category'].values, cmap=cmap, norm=norm,
              origin='upper', interpolation='nearest')
    ax.set_title(f"boundary='{mode}'", fontsize=12)
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
fig.suptitle('Effect of boundary mode on category map', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

With `boundary='nan'`, pixels within the kernel radius of the edge get NaN z-scores and therefore `category=0`. The other modes fill in the edges so every pixel can be classified.

## Working with Dask

For large rasters that don't fit in memory, `emerging_hotspots` supports **Dask-backed DataArrays**. The computation stays lazy until you call `.compute()`. Chunk along the spatial axes only — the time axis should remain in a single chunk (it's typically small: 10–50 steps).

In [ ]:
import dask.array as da

# Chunk spatially (keep full time axis in one chunk)
dask_data = da.from_array(data, chunks=(n_times, 50, 50))
dask_raster = xr.DataArray(dask_data, dims=['time', 'y', 'x'])

# The result is lazy — no computation yet
dask_result = emerging_hotspots(dask_raster, kernel)
print('gi_zscore backing type:', type(dask_result['gi_zscore'].data))

# Trigger computation
dask_result_computed = dask_result.compute()

# Verify it matches the numpy result
np.testing.assert_array_equal(
    dask_result_computed['category'].values,
    result['category'].values,
)
print('Dask result matches numpy result exactly.')

## Summary

`emerging_hotspots` is a single function call that:

1. Computes **Gi\* z-scores** and **confidence bins** at every time step.
2. Applies the **Mann-Kendall trend test** to each pixel's time series.
3. Classifies each pixel into one of **17 trend categories**.

It supports **NumPy**, **CuPy** (GPU), and **Dask** backends, so it scales from small experiments to large satellite image stacks.

### Quick reference

```python
from xrspatial.emerging_hotspots import emerging_hotspots

result = emerging_hotspots(
    raster,              # 3-D DataArray (time, y, x)
    kernel,              # 2-D numpy array (spatial neighbourhood)
    boundary='nan',      # 'nan' | 'nearest' | 'reflect' | 'wrap'
)

result['category']      # (y, x) int8 — trend code -8..8
result['gi_zscore']     # (time, y, x) float32
result['gi_bin']        # (time, y, x) int8
result['trend_zscore']  # (y, x) float32 — Mann-Kendall Z
result['trend_pvalue']  # (y, x) float32 — Mann-Kendall p
```

### References

- Getis, A. and Ord, J. K. (1992). *The analysis of spatial association by use of distance statistics.* Geographical Analysis, 24(3), 189–206.
- Mann, H. B. (1945). *Nonparametric tests against trend.* Econometrica, 13(3), 245–259.
- Kendall, M. G. (1975). *Rank Correlation Methods.* 4th ed. London: Charles Griffin.